## delicious_grab_zone_delta_transformed file upload

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window 
# Load transformed Delta table 
transformed_df = spark.table("delicious_grab_zone_schema.delicious_grab_zone_delta_transformed")

In [0]:
# Monthly revenue
Monthly_revenue = transformed_df.groupBy( 
                F.year("Delivery_date").alias("Year"),
                F.month("Delivery_date").alias("Month")
).agg( 
                F.sum("Total_value").alias("Monthly_revenue") 
).orderBy("Year", "Month")
Monthly_revenue.show()

+----+-----+---------------+
|Year|Month|Monthly_revenue|
+----+-----+---------------+
|2025|    8|           7580|
|2025|    9|           9050|
|2025|   10|           8950|
|2025|   11|           9870|
|2025|   12|          15360|
+----+-----+---------------+



In [0]:
# Save monthly revenue 
Monthly_revenue.write.format("delta").mode("overwrite")\
   .saveAsTable("delicious_grab_zone_schema.Monthly_revenue")

In [0]:
# Top-selling items 
top_items = transformed_df.groupBy("Item") \
      .agg(F.sum("Quantity").alias("Total_sold")) \
      .orderBy(F.desc("Total_sold"))
top_items.show()

+--------------------+----------+
|                Item|Total_sold|
+--------------------+----------+
|          Wheat Loaf|        32|
|Triple Chocolate ...|        25|
|Classic Brownie (...|        22|
|          Multigrain|        20|
|Eggless Classic B...|        19|
|     Nutella Brownie|        17|
|             Blondie|        17|
|          White Loaf|        10|
|Black Forest 1kg ...|         5|
|Oreo Cake 1kg Egg...|         3|
|  Oreo Cake (Janani)|         3|
|Mango Mousse Cake...|         3|
|White Forest Eggl...|         1|
|Mango Cake 1kg (S...|         1|
|Rosemilk Treslech...|         1|
|Black Currant Cak...|         1|
|Oreo Cake 1kg Egg...|         1|
|Black Currant Cak...|         1|
+--------------------+----------+



In [0]:
# Save top items 
top_items.write.format("delta").mode("overwrite")\
    .saveAsTable("delicious_grab_zone_schema.top_items")

In [0]:
# Top customers 
Top_customers = transformed_df.groupBy("Customer_name") \
       .agg(F.sum("Total_value").alias("Customer_spend")) \
       .orderBy(F.desc("Customer_spend"))
Top_customers.show()

+-------------+--------------+
|Customer_name|Customer_spend|
+-------------+--------------+
|      Abejith|          8820|
|     Vasantha|          6540|
|       Dinesh|          4580|
|        Meena|          3620|
|         Ravi|          3380|
|         Arun|          3370|
|      Karthik|          3060|
|       Naveen|          2810|
|       Janani|          2710|
|      Lakshmi|          2650|
|       Sowmya|          2340|
|        Priya|          2220|
|        Divya|          1730|
|        Deepa|          1670|
|     Victoria|          1310|
+-------------+--------------+



In [0]:
# Save top customers 
Top_customers.write.format("delta").mode("overwrite")\
    .saveAsTable("delicious_grab_zone_schema.Top_customers")
          

In [0]:
# Rank customers by spend 
Window_rank = Window.orderBy(F.desc("Customer_spend")) 
Ranked_customers = Top_customers.withColumn("Rank", F.rank().over(Window_rank))
Ranked_customers.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+--------------+----+
|Customer_name|Customer_spend|Rank|
+-------------+--------------+----+
|      Abejith|          8820|   1|
|     Vasantha|          6540|   2|
|       Dinesh|          4580|   3|
|        Meena|          3620|   4|
|         Ravi|          3380|   5|
|         Arun|          3370|   6|
|      Karthik|          3060|   7|
|       Naveen|          2810|   8|
|       Janani|          2710|   9|
|      Lakshmi|          2650|  10|
|       Sowmya|          2340|  11|
|        Priya|          2220|  12|
|        Divya|          1730|  13|
|        Deepa|          1670|  14|
|     Victoria|          1310|  15|
+-------------+--------------+----+



In [0]:
# Save ranked customers 
Ranked_customers.write.format("delta").mode("overwrite")\
    .saveAsTable("delicious_grab_zone_schema.Ranked_customers")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Running totals (revenue over time) 
Window_running = Window.orderBy("Delivery_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
Running_total = transformed_df.withColumn("Running_revenue", F.sum("Total_value").over(Window_running))
Running_total.select("Customer_id", "Delivery_date", "Total_value", "Running_revenue").show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------------+-----------+---------------+
|Customer_id|Delivery_date|Total_value|Running_revenue|
+-----------+-------------+-----------+---------------+
|    ORD0001|   2025-08-01|        750|            750|
|    ORD0002|   2025-08-04|         60|            810|
|    ORD0003|   2025-08-06|        660|           1470|
|    ORD0004|   2025-08-10|        850|           2320|
|    ORD0005|   2025-08-11|        330|           2650|
|    ORD0006|   2025-08-13|        660|           3310|
|    ORD0007|   2025-08-16|       1120|           4430|
|    ORD0008|   2025-08-17|        110|           4540|
|    ORD0009|   2025-08-18|        850|           5390|
|    ORD0010|   2025-08-20|        180|           5570|
|    ORD0011|   2025-08-21|        750|           6320|
|    ORD0012|   2025-08-25|        180|           6500|
|    ORD0013|   2025-08-28|        780|           7280|
|    ORD0014|   2025-08-31|        300|           7580|
|    ORD0015|   2025-09-02|       1050|         

In [0]:
# Save running totals 
Running_total.write.format("delta").mode("overwrite") \
     .saveAsTable("delicious_grab_zone_schema.Running_totals")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Invalid payment modes 
Invalid_payment = transformed_df.withColumn(
    "Invalid_Payment_Flag",
    F.when(~F.col("Mode_of_payment").isin("Cash", "Card", "UPI"), 1).otherwise(0)
)
Invalid_payment.show()


+-----------+-------------+-------------+--------------------+--------+-----+---------+---------------+-----------+--------+-----------+---------------+--------------------+
|Customer_id|Delivery_date|Customer_name|                Item|Quantity|Price|  Address|Mode_of_payment|Total_value|Category|Offer_Price|Bulk_Order_Flag|Invalid_Payment_Flag|
+-----------+-------------+-------------+--------------------+--------+-----+---------+---------------+-----------+--------+-----------+---------------+--------------------+
|    ORD0001|   2025-08-01|      Karthik|Eggless Classic B...|       3|  250|     E101|           Cash|        750| Brownie|      225.0|             No|                   0|
|    ORD0002|   2025-08-04|     Victoria|          Wheat Loaf|       1|   60|     A301|           Cash|         60|   Bread|       60.0|             No|                   0|
|    ORD0003|   2025-08-06|       Naveen|Classic Brownie (...|       3|  220|     I116|           Cash|        660| Brownie|      

In [0]:
# Save invalid payments 
Invalid_payment.write.format("delta").mode("overwrite") \
     .saveAsTable("delicious_grab_zone_schema.Invalid_payment")

In [0]:
# Duplicate orders 
duplicates = transformed_df.groupBy("Customer_id")\
    .count()\
    .filter(F.col("count") > 1)
duplicates.show()

+-----------+-----+
|Customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# Save duplicates 
duplicates.write.format("delta").mode("overwrite") \
      .saveAsTable("delicious_grab_zone_schema.duplicate_orders")

In [0]:
transformed_df.display()

Customer_id,Delivery_date,Customer_name,Item,Quantity,Price,Address,Mode_of_payment,Total_value,Category,Offer_Price,Bulk_Order_Flag
ORD0001,2025-08-01,Karthik,Eggless Classic Brownie,3,250,E101,Cash,750,Brownie,225.0,No
ORD0002,2025-08-04,Victoria,Wheat Loaf,1,60,A301,Cash,60,Bread,60.0,No
ORD0003,2025-08-06,Naveen,Classic Brownie (4pcs),3,220,I116,Cash,660,Brownie,198.0,No
ORD0004,2025-08-10,Vasantha,Black Forest 1kg (Victoria),1,850,H310,Cash,850,Other,850.0,No
ORD0005,2025-08-11,Ravi,Blondie,3,110,B107,Card,330,Other,110.0,No
ORD0006,2025-08-13,Abejith,Classic Brownie (4pcs),3,220,A311,Cash,660,Brownie,198.0,No
ORD0007,2025-08-16,Meena,Mango Mousse Cake (Dinesh),1,1120,J310,Cash,1120,Cake,1120.0,No
ORD0008,2025-08-17,Vasantha,White Loaf,1,110,A202,Cash,110,Bread,110.0,No
ORD0009,2025-08-18,Vasantha,Black Forest 1kg (Victoria),1,850,F104,Cash,850,Other,850.0,No
ORD0010,2025-08-20,Arun,Wheat Loaf,3,60,E207,Card,180,Bread,60.0,No
